# Inference Rules!

### Some quick logistics
* Problemsets:
    * PS2 is due Wednesday
    * PS3 will be available tomorrow; due on next Monday at midnight
* The project is available and due next Friday at midnight
* Teaching assignments:
    * First group will be going this Thursday
    * Second group will be going next Thursday
    * If you don't have a group, LET ME KNOW

### The plan for this week
* Monday: <-- YOU ARE HERE
    * Review of OOP and Pattern Matching
    * Abstract/Concrete Syntax
    * Intro to inference rules
* Tuesday: 
    * Introduction to Lettuce 
    * Inference rules continued
* Recitation: Project overview
* Thursday:
    * First student lecture!
    * Discussing variables and the environment

### The rough overview of today's lecture
* Some review
* Concrete vs Abstract Syntax
* Inference Rule Terminology
* Inference Rules in Practice

### Review of OOP and Pattern Matching
Suppose I wanted to implement BNF Grammar that looked like the following:
$$\begin{array}{lll}
Polygon & \Rightarrow & Rectangle(d, d) \\
 &|& Triangle(d, d) \\
 \\
 &&\texttt{d is a Scala Double}
\end{array}$$

In the case of a rectangle, the two variables are the height and length.  In the case of the triangle, we can assume they are the base and the height.

What would this look like in object-oriented code?

In [ ]:
sealed trait Polygon{
    def getArea(): Double
}

case class Rectangle(height: Double, length: Double) extends Polygon {
    def getArea(): Double = {
        height * length
    }
}
case class Triangle(base: Double, height: Double) extends Polygon {
    def getArea(): Double = {
        base * height * 0.5
    }
}

What would it look like to implement a getArea() function for all Polygons?  Try it with the visitor pattern.

Try implementing the same method, but with pattern matching instead.

In [ ]:
sealed trait PolygonPattern {
    def getArea(): Double = {
        this match {
            case RectanglePattern(h, l) => h * l
            case TrianglePattern(base, height) => base * height * 0.5
        }
    }
}

case class RectanglePattern(height: Double, length: Double) extends PolygonPattern
case class TrianglePattern(base: Double, height: Double) extends PolygonPattern

# Concrete vs Abstract Syntax

## Syntax vs Semantics
* Semantics is the actual meaning behind words
* Syntax is the structure of a given language
* One is useless without the other . . . but more often than not, we have an idea of what we mean, so we tend to focus a lot more on syntax
* Consider human language
    * there are so many human languages
    * they each have shared semantics (broadly)
    * but the syntax can vary a lot
    * e.g. 
        * English: "My name is Kevin"
        * German: "Ich heisse Kevin"

## Abstract Syntax
So far we've worked mostly on abstract syntax
* because it's easier to code
    * directly translates to Scala~ish
* and I find it's easier for students to understand

Consider the abstract BNF grammar above for our list:

$$\begin{array}{lll}
MyList & \Rightarrow & EmptyList \\
 &|& ListNode(d, MyList) \\
 \\
 &&\texttt{d is a Scala Double}
\end{array}$$

What code would it result in?

In [ ]:
sealed trait MyList
case object EmptyList extends MyList
case class ListNode(h: Double, t: MyList) extends MyList

## Concrete Syntax
But as we move forward in the class we want to consider concrete syntax, because it is agnostic of the Scala programming language.  So we can more easily apply our knowledge to other tech stacks

By definition, our concrete syntax determines how a language looks like to humans (i.e. programmars).  An abstract syntax determines how a language looks like to a compiler/interpreter.  So one of these things is limited to a specific programming language, and one of them is not.

Consider the concept of a list. The real lists that I may be interested in might look like
* X
* 1 $\rightarrow$ X
* 5 $\rightarrow$ 10 $\rightarrow$ 15 $\rightarrow$ X
* 24 $\rightarrow$ 5 $\rightarrow$ 3155 $\rightarrow$ X
    * perhaps they don't need to be sorted
    
* here it is as a concrete BNF grammar with start symbol `l`

$$\begin{array}{lll}
l & \Rightarrow & X \\
&|& n\;\rightarrow\;l \\
\\
&&n \in \mathbb{N}
\end{array}$$

* NOTE: $\rightarrow$ and $\Rightarrow$ have different meanings
    * $\rightarrow$ is a concrete terminal of the object language of lists
        * Keep in mind, this is context free language
        * The symbol really just lets us describe the order of a list
    * $\Rightarrow$ is a meta-symbol of BNF grammars
    

So we've got an ability to define classes/traits based on our grammar.  But what about the functionality, the meaning, behind these traits and classes?

# Inference Rules
* Enter 
    * the rules of inference
    * inference rules
* when combined with grammars and judgement form, we have an operational semantics for some language
    * Judgement forms are just assertions made on top of syntax (easier to see as an example)

## Terminology
* Inference rules describe the relation between a set of judgments
    * take the following form: its a line with stuff below it and above it
    
$$\begin{array}{c}
premise_n,\;\;\;\dots,\;\;\;premise_n
\\ \hline
conclusion
\end{array}\texttt{rule-name}$$

* Conclusion: the conclusion is listed below the line and is read first
* Premise: premise are written above the line. An inference rule can have 0-or-many premise
    * Axiom: an inference rule without any premise is called an axiom

## Example: Prepend to a list
* Concrete Grammar

$$\begin{array}{lll}
l & \Rightarrow & X \\
&|& n\;\rightarrow\;l \\
\\
&&n \in \mathbb{N}
\end{array}$$

* Judgment form: `l prepend n = l'`

* Inference rules:

$$\begin{array}{c}
\\ \hline
X\;prepend\;n = n\rightarrow X
\end{array}\texttt{prepend-empty-list}$$

* read: Given a list of form `X` and a number `n`, `X prepend n` shall yield a new list `n`$\rightarrow$`X`
* This has a conclusion
* this has no premise
    * this is an axiom

$$\begin{array}{c}
l \neq X
\\ \hline
l\;prepend\;n = n\;\rightarrow\;l
\end{array}\texttt{prepend-non-empty-list}$$

* read: Given a list `l` and a number `n`, `l prepend n` shall yield a new list `n`$\rightarrow$`l`, if and only if list `l` is not the empty list `X`
* This has a conclusion
* this has one premise
    * this is NOT an axiom

In [ ]:
// AS CODE
sealed trait MyList {
    def prepend(d: Double): MyList = {
        this match {
            case EmptyList => ListNode(d, EmptyList)
            case l@ListNode(head, tail) => ListNode(d, l)
        }
    }
}
case object EmptyList extends MyList
case class ListNode(head: Double, tail: MyList) extends MyList

* we could condence to a single rule

$$\begin{array}{c}
\\ \hline
l\;prepend\;n = n\rightarrow l
\end{array}\texttt{prepend-empty-list}$$

* read: Given a list `l` of any form and a number `n`, `l prepend n` shall yield a new list `n`$\rightarrow$`l`
* This has a conclusion
* this has no premise
    * this is an axiom

In [ ]:
// AS CODE
sealed trait MyList {
    def prepend(d: Double): MyList = {
        ListNode(d, this)
    }
}
case object EmptyList extends MyList
case class ListNode(head: Double, tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
assert(t0.prepend(10) == t1)
assert((t1 prepend 5) == t2)

## Example: Append to a list
* Concrete Grammar

$$\begin{array}{lll}
l & \Rightarrow & X \\
&|& n\;\rightarrow\;l \\
\\
&&n \in \mathbb{N}
\end{array}$$

* Judgment form: `l append n = l'`

* Inference rules:

$$\begin{array}{c}
\\ \hline
X\;append\;n = n\rightarrow X
\end{array}\texttt{append-empty-list}$$

* read: Given a list of form `X` and a number `n`, `X append n` shall yield a new list `n`$\rightarrow$`X`
* This has a conclusion
* this has no premise
    * this is an axiom

$$\begin{array}{c}
l\;append\;n_2 = l_{New},
\\ \hline
n_1\rightarrow l\;append\;n_2 = n_1\;\rightarrow\;l_{New}
\end{array}\texttt{append-non-empty-list}$$

* read: Given a list `n1`$\rightarrow$`l` and a number `n2`, `n1`$\rightarrow$`l append n2` shall yield a new list `n1`$\rightarrow$`lNew`, if and only if `l append n2 = lNew`
* This has a conclusion
* this has one premise
    * this is NOT an axiom
    * this is inductively defined
        * in the premise of a rule for `append`
        * we use `append` itself

In [ ]:
// AS CODE
sealed trait MyList {
    def prepend(d: Double): MyList = {
        ListNode(d, this)
    }
    
    def append(d: Double): MyList = {
        this match {
            case EmptyList => ListNode(d, EmptyList)
            case ListNode(n1, l) => {
                // Append d onto tail
                val lNew = l.append(d)
                
                // create new listnode with n1 -> new tail
                ListNode(n1, lNew)
            }
        }
    }
}
case object EmptyList extends MyList
case class ListNode(head: Double, tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
val t3 = ListNode(5, ListNode(10, ListNode(15, t0)))
assert(t0.prepend(10) == t1)
assert((t1 prepend 5) == t2)
assert((t2 append 15) == t3)

## Example: Inserting into an ordered list
* Insert into a list
    * suppose the list is going to be ordered smallest to largest
    * Let us implement an insert method
    * to insert a new number into the list
    * in it's appropriate place given the current order of the list
    
* Concrete Grammar

$$\begin{array}{lll}
l & \Rightarrow & X \\
&|& n\;\rightarrow\;l \\
\\
&&n \in \mathbb{N}
\end{array}$$

* Judgment form: l insert n = l'

* Inference Rules
    * rules of thumb, how many rules will we need
        * for this judgment form?
        * for the input grammar(s)?

~~~
P1,     P2,     ...
-----------------------------------(insert-???)
    Conclusion
~~~

In [ ]:
// AS CODE
sealed trait MyList {
    def prepend(d: Double): MyList = {
        ListNode(d, this)
    }
    
    def append(d: Double): MyList = {
        this match {
            case EmptyList => ListNode(d, EmptyList)
            case ListNode(n1, l) => {
                val lNew = l.append(d)
                ListNode(n1, lNew)
            }
        }
    }
    
    def insert(d: Double): MyList = {
        this match {
            ???
        }
    }
    
    
}
case object EmptyList extends MyList
case class ListNode(head: Double, tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
val t3 = ListNode(5, ListNode(10, ListNode(15, t0)))
assert(t0.prepend(10) == t1)
assert((t1 prepend 5) == t2)
assert((t2 append 15) == t3)

## Delete
* Delete an element of the list
    * given a list `l`
    * and a number `n`
    * Delete the first found instance of number `n`
    * If the number `n` is never found, return `ERROR`
    * Dealers choice: You can assume the list is ordered, or not. Please state you assumptions clearly
    
* Concrete Grammar **`s`**

$$\begin{array}{lll}
l & \Rightarrow & X \\
&|& n\;\rightarrow\;l \\
\\
v&\Rightarrow& n \\
&|& ERROR \\
\\
&&n \in \mathbb{N}
\end{array}$$

* Judgment form: `???`

* Assumptions:
    * Dealers choice: You can assume the list is ordered, or not. Please state you assumptions clearly
    * ???
    
    
* Inference Rules

~~~
P1,     P2,     ...
-----------------------------------(insert-???)
    Conclusion
~~~

In [ ]:
// AS CODE
sealed trait MyList {
    def prepend(d: Double): MyList = {
        ListNode(d, this)
    }
    
    def append(d: Double): MyList = {
        this match {
            case EmptyList => ListNode(d, EmptyList)
            case ListNode(n1, l) => {
                val lNew = l append d
                ListNode(n1, lNew)
            }
        }
    }
    
    def insert(d: Double): MyList = {
        this match {
            case EmptyList => ListNode(d, EmptyList)
            case ListNode(head, _) if d <= head => {
                ListNode(d, this)
            }
            case ListNode(head, tail) => {
                val newTail = tail insert d
                ListNode(head, newTail)
            }
        }
    }
    
    def delete(d: Double): MyList = {
        // _YOUR_CODE_HERE
        ???
    }
}
case object EmptyList extends MyList
case class ListNode(head: Double, tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
val t3 = ListNode(5, ListNode(10, ListNode(15, t0)))
assert(t0.prepend(10) == t1)
assert((t1 prepend 5) == t2)
assert((t2 append 15) == t3)

val t4 = ListNode(5, ListNode(7, ListNode(10, ListNode(15, t0))))
assert((t3 insert 7) == t4)


// _YOUR_TESTS_HERE

// // UNCOMMENT WHEN READY TO TEST
// assert((t4 delete 7) == t3)
// assert((t3 delete 15) == t2)
// assert((t2 delete 5) == t1)
// assert((t1 delete 10) == t0)

// try {
//     t4 delete 3155
//     println("shouldn't see this message")
// } catch {
//     case _: Throwable => assert(true)
// }

# Brining it to the PL world
* We define expressions of some domain specific language (DSL)
    * Here, over lists and operations on lists
    * isValid, sum, insert, delete, +
* We define the possible values
    * Here, numbers, boolean, lists, ERROR
* We define how to evaluate each expression to a value
    * e $\Downarrow$ v
        * authors a 'big-step' interpreter
        * expression e evaluates in 0-or-many steps to a value v
        * **`eval(e) = v`**
            * alternate notation
            * expression e evaluates in 0-or-many steps to a value v
            * used in this course
        * e.g.
            * `1 + 2 * 3` $\Downarrow$ `7`
            * OR
            * `eval(1 + 2 * 3) = 7`
    * e $\longrightarrow$* v
        * expression e steps in 0-or-many steps to a value v
        * authors a 'small-step' interpreter
        * supported by additional judgment form: 
            * e $\longrightarrow$ e'
            * expression e steps to new expression e' in exactly one step
        * e.g.
            * `1 + 2 * 3` $\longrightarrow$* `7`
                * BECAUSE
                * `1 + 2 * 3` $\longrightarrow$ `1 + 6`
                * `1 + 6` $\longrightarrow$ `7`
* this get's tricky
* we'll attempt this
* if we get too confused, we can back off

## For our language of lists `l`
* grammar

$$\begin{array}{lll}
e &\Rightarrow& v \\
&|& sum(e) \\
&|& e_1 + e_2 & \texttt{add e2 to each element of e1}\\
&|& e_1 \leftarrow e_2 & \texttt{insert e2 to e1}\\
&|& isValid(e) \\
&|& e_1\;delete\;e_2 \\
&|& e_1 \rightarrow e_2 \\
\\
v & \Rightarrow & vl \\
& | & b \\
&|& n \\
&|& ERROR \\
\\
vl &\Rightarrow& X \\
&|& n \rightarrow vl \\
\\
&& \texttt{n is a number} \\
&& \texttt{b is a boolean} \\
\end{array}$$

In [ ]:
// let's agree on code for above

// sum(e)
// e1 + e2
// e1 <- e2
// isValid(e)
// e1 delete e2
// e1 -> e2

// vl
// n
// b
// error

* judgment form: `eval(e) = v`

In [ ]:
// define the function stub

* Inference Rules

$$\begin{array}{c}
\\ \hline
eval(v) = v
\end{array}\texttt{value-ok}$$
<br /><br /><br />

In [ ]:
// impl

$$\begin{array}{c}
eval(e) = v,\;\;\;v \notin vl
\\ \hline
eval(sum(e)) = ERROR
\end{array}\texttt{sum-ERROR}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e) = X
\\ \hline
eval(sum(e)) = 0
\end{array}\texttt{sum-empty-list}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e) = n_1 \rightarrow l,\;\;\;sum(l) = n_2,\;\;\;n_1 + n_2 = n
\\ \hline
eval(sum(e)) = n
\end{array}\texttt{sum-NON-empty-list}$$
<br /><br /><br />

In [ ]:
// impl

$$\begin{array}{c}
eval(e_1) = v_1,\;\;\;v_1 \notin vl
\\ \hline
eval(e_1 + e_2) = ERROR
\end{array}\texttt{+-ERROR-1}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e_1) = vl,\;\;\;eval(e_2) = v_2\;\;\;v_2 \notin \mathbb{N}
\\ \hline
eval(e_1 + e_2) = ERROR
\end{array}\texttt{+-ERROR-2}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e1) = X,\;\;\;eval(e_2) = n_2
\\ \hline
eval(e_1 + e_2) = X
\end{array}\texttt{+-empty-list}$$
<br /><br /><br />


$$\begin{array}{c}
eval(e1) = n_1 \rightarrow vl_1,\;\;\;eval(e_2) = n_2,\;\;\;n_1 + n_2 = n_3,\;\;\;eval(vl_1 + n_2) = vl_2
\\ \hline
eval(e_1 + e_2) = n_3 + vl_2
\end{array}\texttt{+-NOT-empty-list}$$
<br /><br /><br />

In [ ]:
// impl

$$\begin{array}{c}
eval(e_1) = v_1,\;\;\;v_1 \notin vl
\\ \hline
eval(e_1 \leftarrow e_2) = ERROR
\end{array}\texttt{insert-ERROR-1}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e_1) = vl,\;\;\;eval(e_2) = v_2\;\;\;v_2 \notin n
\\ \hline
eval(e_1 \leftarrow e_2) = ERROR
\end{array}\texttt{insert-ERROR-2}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e1) = X,\;\;\;eval(e_2) = n_2
\\ \hline
eval(e_1 \leftarrow e_2) = n_2 \rightarrow X
\end{array}\texttt{insert-empty-list}$$
<br /><br /><br />


$$\begin{array}{c}
eval(e1) = n_1 \rightarrow vl_1,\;\;\;eval(e_2) = n_2,\;\;\;n_2 \leq n_1
\\ \hline
eval(e_1 \leftarrow e_2) = n_2 \rightarrow n_1 \rightarrow vl_1
\end{array}\texttt{insert-NOT-empty-list-here}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e1) = n_1 \rightarrow vl_1,\;\;\;eval(e_2) = n_2,\;\;\;n_2 \gt n_1,\;\;\;eval(vl_1 \leftarrow n_2) = vl_2
\\ \hline
eval(e_1 \leftarrow e_2) = n_1 \rightarrow vl_2
\end{array}\texttt{insert-NOT-empty-list-there}$$
<br /><br /><br />

In [ ]:
// impl

$$\begin{array}{c}
eval(e_1) = v_1,\;\;\;v_1 \notin vl
\\ \hline
eval(isValid(e_1)) = ERROR
\end{array}\texttt{isValid-ERROR-1}$$
<br /><br /><br />


$$\begin{array}{c}
eval(e_1) = X
\\ \hline
eval(isValid(e_1)) = true
\end{array}\texttt{isValid-empty-list}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e_1) = n_1 \rightarrow X,
\\ \hline
eval(isValid(e_1)) = true
\end{array}\texttt{isValid-len1-list}$$
<br /><br /><br />


$$\begin{array}{c}
eval(e_1) = n_1 \rightarrow n_2 \rightarrow vl_3,\;\;\;n_1 \leq n_2,\;\;\;eval(isValid(n_2 \rightarrow vl_3)) = b
\\ \hline
eval(isValid(e_1)) = b
\end{array}\texttt{isValid-lenn-list-b}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e_1) = n_1 \rightarrow n_2 \rightarrow vl_3,\;\;\;n_1 \gt n_2
\\ \hline
eval(isValid(e_1)) = false
\end{array}\texttt{isValid-lenn-list-false}$$
<br /><br /><br />


In [ ]:
// impl

$$\begin{array}{c}
eval(e_1) = v_1,\;\;\;v_1 \notin \mathbb{N}
\\ \hline
eval(e_1 -> e_2) = ERROR
\end{array}\texttt{prepend-ERROR-1}$$
<br /><br /><br />


$$\begin{array}{c}
eval(e_1) = n_1,\;\;\;eval(e_2)=v_2,\;\;\;v_2 \notin vl
\\ \hline
eval(e_1 -> e_2) = ERROR
\end{array}\texttt{prepend-ERROR-2}$$
<br /><br /><br />

$$\begin{array}{c}
eval(e_1) = n_1,\;\;\;eval(e_2)=vl_2
\\ \hline
eval(e_1 -> e_2) = n_1 -> vl_2
\end{array}\texttt{prepend-ok}$$
<br /><br /><br />


In [ ]:
// impl